# HTTP and JSON

A website, a mobile app and a script all talk to a server the same way: they
send an HTTP **request**, the server sends back a **response**.

| Request | Example |
|---|---|
| method | `GET` read, `POST` create, `PUT` replace, `DELETE` delete |
| path | `/api/shows/le-kiosque-2027-06-11-2330` |
| query parameters | `?day=2027-06-11&venue=le-kiosque` |
| body (POST, PUT) | JSON: `{"title": "Late Laughs", "duration": 60}` |

| Response | Example |
|---|---|
| status code | `200` OK, `201` created, `204` nothing to send back |
| | `404` not found, `409` conflict, `422` invalid data, `500` server bug |
| body | JSON |

Codes starting with 2 are successes, 4 are the client's mistakes, 5 the
server's.

## JSON: Python data as text

JSON is the text format of web APIs. It has objects (Python `dict`), arrays
(`list`), strings, numbers, `true` / `false` and `null` (`None`).

In [1]:
import json

show = {"title": "Late Laughs", "duration": 60, "price": 12.0, "sold_out": False}
text = json.dumps(show)
print(text)
print(type(text))

{"title": "Late Laughs", "duration": 60, "price": 12.0, "sold_out": false}
<class 'str'>


In [2]:
back = json.loads(text)
print(back["duration"] + 30)
print(type(back))

90
<class 'dict'>


JSON has no date type. A `datetime` must be turned into text first:

In [3]:
from datetime import datetime

json.dumps({"start": datetime(2027, 6, 11, 23, 30)})

TypeError: Object of type datetime is not JSON serializable

In [4]:
start = datetime(2027, 6, 11, 23, 30)
print(json.dumps({"start": start.isoformat(timespec="minutes")}))
print(datetime.fromisoformat("2027-06-11T23:30"))

{"start": "2027-06-11T23:30"}
2027-06-11 23:30:00


## A first server

FastAPI turns Python functions into HTTP routes. `@app.get("/")` means:
*when a `GET` request arrives on `/`, call this function*. The returned
`dict` is sent as JSON.

In [5]:
from fastapi import FastAPI

app = FastAPI()


@app.get("/")
def home() -> dict:
    return {"message": "Welcome to the Deauville Festival du Rire."}

To call the application without starting a server, `TestClient` sends it
requests from Python. The tests of the course, and yours, use it.

In [6]:
from fastapi.testclient import TestClient

client = TestClient(app)
response = client.get("/")
print(response.status_code)
print(response.headers["content-type"])
print(response.json())

200
application/json
{'message': 'Welcome to the Deauville Festival du Rire.'}


In [7]:
response = client.get("/programme")
print(response.status_code)
print(response.json())

404
{'detail': 'Not Found'}


No route for `/programme`: FastAPI answers `404` by itself.

To see it in a browser, the same application is started by a web server,
uvicorn: see [hello_api.py](hello_api.py) and [api.md](../api.md).